# Week 6 Practice: CNN Foundations

CSCI1409 Deep Learning · based on Lecture 5

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Task 1: Convolution

A convolution slides a small kernel over the input. At each position it multiplies the matching entries, sums them and adds a bias. The same kernel is used at every position.

We use the example from the lecture:

$$X = \begin{bmatrix} 1 & 2 & 0 & 1 \\ 0 & 1 & 3 & 2 \\ 2 & 2 & 1 & 0 \\ 1 & 0 & 2 & 3 \end{bmatrix}, \qquad K = \begin{bmatrix} 1 & 0 & -1 \\ 1 & 0 & -1 \\ 1 & 0 & -1 \end{bmatrix}$$

### 1a) One convolution by hand

1. Compute all four entries of $Y$ on paper (stride 1, padding 0, bias 0). Write the calculation for each entry.
2. Check your answer with `F.conv2d`. It expects the input as `[B, C, H, W]` and the kernel as `[C_out, C_in, k, k]`, so use `.view(1, 1, 4, 4)` and `.view(1, 1, 3, 3)`.
3. What would $Y$ be with bias $b = 1$?

**Your answer:**

Y(0,0) = (1 + 0 + 0) + (0 + 0 - 3) + (2 + 0 - 1) = 1 - 3 + 1 = -1

Y(0,1) = (2 + 0 - 1) + (1 + 0 - 2) + (2 + 0 + 0) = 1 - 1 + 2 = 2

Y(1,0) = (0 + 0 - 3) + (2 + 0 - 1) + (1 + 0 - 2) = -3 + 1 - 1 = -3

Y(1,1) = (1 + 0 - 2) + (2 + 0 + 0) + (0 + 0 - 3) = -1 + 2 - 3 = -2

In [2]:
X = torch.tensor([[1., 2., 0., 1.],
                  [0., 1., 3., 2.],
                  [2., 2., 1., 0.],
                  [1., 0., 2., 3.]])

K = torch.tensor([[1., 0., -1.],
                  [1., 0., -1.],
                  [1., 0., -1.]])


X_4d = X.view(1, 1, 4, 4)
K_4d = K.view(1, 1, 3, 3)

Y = F.conv2d(X_4d, K_4d, bias=None, stride=1, padding=0)

print(Y.squeeze())

tensor([[-1.,  2.],
        [-3., -2.]])


### 1b) Output size and parameters

From the lecture:

$$H_{out} = \left\lfloor \frac{H + 2p - k}{s} \right\rfloor + 1, \qquad \text{params} = C_{out}\,(C_{in}\,k^2 + 1)$$

| Case | Input | Filters | $k$ | $p$ | $s$ |
|---|---|---|---|---|---|
| A | 1×28×28 | 8 | 3 | 0 | 1 |
| B | 1×28×28 | 8 | 3 | 1 | 1 |
| C | 1×28×28 | 8 | 3 | 1 | 2 |
| D | 3×32×32 | 16 | 5 | 2 | 1 |

1. Use the formulas to compute the output shape and the number of parameters for each case.
2. Check with `nn.Conv2d`. Pass `torch.randn(1, C_in, H, H)` through the layer, print the output shape, and count the parameters with `sum(p.numel() for p in layer.parameters())`.
3. A, B and C have the same number of parameters. Why?

In [3]:
cases = [
    ('A', 1, 28, 8, 3, 0, 1),
    ('B', 1, 28, 8, 3, 1, 1),
    ('C', 1, 28, 8, 3, 1, 2),
    ('D', 3, 32, 16, 5, 2, 1)
]

for name, c_in, h, c_out, k, p, s in cases:
    layer = nn.Conv2d(in_channels=c_in, out_channels=c_out, kernel_size=k, stride=s, padding=p)
    
    x = torch.randn(1, c_in, h, h)
    
    output = layer(x)
    
    num_params = sum(p.numel() for p in layer.parameters())
    
    print(f"Case {name}: Output shape = {list(output.shape)}, Parameters = {num_params}")

Case A: Output shape = [1, 8, 26, 26], Parameters = 80
Case B: Output shape = [1, 8, 28, 28], Parameters = 80
Case C: Output shape = [1, 8, 14, 14], Parameters = 80
Case D: Output shape = [1, 16, 32, 32], Parameters = 1216


**Your answer:**

1) 
params = Cout(Cin * k^2 + 1)

A = 8(3^2 + 1) = 80

B = 8(3^2 + 1) = 80

C = 8(3^2 + 1) = 80

D = 16(3 * 5^2 + 1) = 1216

2) 

3) Because by formula A, B, C depend only from Cout, Cin, k - kernel size


## Task 2: ReLU, pooling and the full CNN

### 2a) ReLU and max pooling

$Y$ is your result from 1a. $P$ is the matrix from the max pooling slide:

$$P = \begin{bmatrix} 1 & 3 & 2 & 0 \\ 4 & 2 & 1 & 5 \\ 0 & 1 & 3 & 2 \\ 2 & 0 & 1 & 4 \end{bmatrix}$$

1. Predict $\text{ReLU}(Y)$ and 2×2 max pooling with stride 2 of $P$ by hand.
2. Check with `nn.ReLU()` and `nn.MaxPool2d(2)`.
3. A batch of shape B×8×28×28 goes through `nn.MaxPool2d(2)`. What is the output shape? Why does the number of channels not change?

**Your answer:**

In [4]:
P = torch.tensor([[1., 3., 2., 0.],
                  [4., 2., 1., 5.],
                  [0., 1., 3., 2.],
                  [2., 0., 1., 4.]])

Y = torch.tensor([[-1., 2.], [-3., -2.]])
relu = nn.ReLU()
print("ReLU(Y):")
print(relu(Y))

P_4d = P.view(1, 1, 4, 4)
max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
output_pool = max_pool(P_4d)

print("MaxPool(P):")
print(output_pool.squeeze())

ReLU(Y):
tensor([[0., 2.],
        [0., 0.]])
MaxPool(P):
tensor([[4., 5.],
        [2., 4.]])


1) MaxPooling = [[4, 5], 
                 [2, 4]] 

3) B * 8 * 28 * 28; The pooling operation does not affect the number of channels C.  

### 2b) The lecture CNN

1. Build the CNN from Lecture 5 (slide "The CNN in PyTorch") with `nn.Sequential`. Pass `x = torch.randn(4, 1, 28, 28)` through it one layer at a time and print the shape after each layer. Hint: `for layer in model: x = layer(x)`.
2. Count the parameters of the whole model and of each layer. What is the flattened feature count? Which layer has the most parameters?
3. An MLP for the same task (Flatten, Linear(784, 128), ReLU, Linear(128, 10)) has 101,770 parameters. Why does the CNN need so many fewer?

In [5]:
model = nn.Sequential(
    nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nn.Flatten(),
    nn.Linear(8 * 14 * 14, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)

x = torch.randn(4, 1, 28, 28)
for i, layer in enumerate(model):
    x = layer(x)
    print(f"Layer {i} ({type(layer).__name__}): {list(x.shape)}")

total_params = 0
for i, layer in enumerate(model):
    layer_params = sum(p.numel() for p in layer.parameters())
    total_params += layer_params
    if layer_params > 0:
        print(f"Layer {i} ({type(layer).__name__}): {layer_params} params")

print(f"\ntotal: {total_params}")

Layer 0 (Conv2d): [4, 8, 28, 28]
Layer 1 (ReLU): [4, 8, 28, 28]
Layer 2 (MaxPool2d): [4, 8, 14, 14]
Layer 3 (Flatten): [4, 1568]
Layer 4 (Linear): [4, 128]
Layer 5 (ReLU): [4, 128]
Layer 6 (Linear): [4, 10]
Layer 0 (Conv2d): 80 params
Layer 4 (Linear): 200832 params
Layer 6 (Linear): 1290 params

total: 202202


**Your answer:**

## Stretch (optional): convolution with loops

Write `conv2d_manual(X, K)` with two `for` loops (stride 1, padding 0, no bias). At output position $(i, j)$ take the patch `X[i:i+k, j:j+k]`, multiply it with K element by element and sum. Check that you get the same $Y$ as in 1a.

In [6]:
# Your code here